# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY') 

In [3]:
# Validating the presence of the API key (optional)
if not OPENAI_API_KEY:
    raise ValueError("⚠️ OPENAI_API_KEY not found. Please create a .env file with your key.")
else:
    print("✅ OPENAI_API_KEY loaded successfully.")


✅ OPENAI_API_KEY loaded successfully.


### Example 1

#### Create our QandA application

In [5]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
# from langchain.llms import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain.document_loaders import CSVLoader, TextLoader
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain.indexes import VectorstoreIndexCreator
# from langchain.vectorstores import DocArrayInMemorySearch
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain.chains import LLMChain


In [6]:
# Loading the CSV file
file = 'data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding='utf-8') # adding encoding to avoid errors
data = loader.load()
print(f"Loaded {len(data)} documents from the CSV file.") # verification step

Loaded 1000 documents from the CSV file.


In [9]:
# !pip install --upgrade --force-reinstall sentence-transformers

In [7]:
# Creating the vector store index
from langchain.text_splitter import RecursiveCharacterTextSplitter # to avoid errors with large documents

# Splitting the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)

# Creating the embedding model
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", # specify the model name
    model_kwargs={'device': 'cpu'}  # use 'cuda' if GPU is available
)

# Building the vector index
index = VectorstoreIndexCreator( 
    text_splitter=text_splitter,
    embedding=emb,
    vectorstore_cls=DocArrayInMemorySearch,
).from_loaders([loader])

# Verification step
print("Vectorstore index created successfully!")

/opt/anaconda3/envs/langchain_rag/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/opt/anaconda3/envs/langchain_rag/lib/python3.11/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


Vectorstore index created successfully!


In [8]:
# Setting up the RetrievalQA chain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0) # specify the model name
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(search_kwargs={"k": 4}), # specify number of documents to retrieve
    return_source_documents=True, # specify to return source documents
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [ ]:
# Testing the QA system
data[10]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [ ]:
# Testing with another document
data[11]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [11]:
from langchain.prompts import PromptTemplate
from langchain.schema import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0) # specify the model name

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = llm_chain.invoke({"query": query})["text"] # using .invoke

# Print the result
print(result)


/var/folders/xx/q0js_gcd29lfhsshk693bdyw0000gn/T/ipykernel_59891/1102804752.py:46: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


answer='Yes, the Cozy Comfort Pullover Set is available in different colors.'


#### LLM-Generated examples

In [12]:
from langchain.evaluation.qa import QAGenerateChain

In [13]:
# Creating the example generation chain
example_gen_chain = QAGenerateChain.from_llm(
    ChatOpenAI(model="gpt-4o-mini", temperature=0.0)  # specify the model name
)

In [14]:
# Initializing the LLM
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [15]:
# Generating new examples
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]] # using first 5 documents
)

/var/folders/xx/q0js_gcd29lfhsshk693bdyw0000gn/T/ipykernel_59891/2308712842.py:2: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(


In [ ]:
# Displaying the first generated example
new_examples[0]

{'qa_pairs': {'query': "What are the key features of the Women's Campside Oxfords, including materials, construction, and fit recommendations?",
  'answer': "The Women's Campside Oxfords are designed with a super-soft canvas material that provides a broken-in feel from the first wear. They feature thick cushioning and quality construction for comfort. The fit recommendation is to order your regular shoe size, and for half sizes not offered, to order up to the next whole size. The shoes have an EVA innersole with Cleansport NXT® antimicrobial odor control, a vintage hunt, fish, and camping motif on the innersole, and a moderate arch contour. Additionally, they include an EVA foam midsole for cushioning and support, and a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern. The approximate weight is 1 lb. 1 oz. per pair."}}

In [ ]:
# Displaying the first document
data[0]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \n\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \n\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \n\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \n\nQuestions? Please contact us for any inquiries.")

In [ ]:
# Flattening the generated QA pairs
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': "What are the key features of the Women's Campside Oxfords, including materials, construction, and fit recommendations?",
  'answer': "The Women's Campside Oxfords are designed with a super-soft canvas material that provides a broken-in feel from the first wear. They feature thick cushioning and quality construction for comfort. The fit recommendation is to order your regular shoe size, and for half sizes not offered, to order up to the next whole size. The shoes have an EVA innersole with Cleansport NXT® antimicrobial odor control, a vintage hunt, fish, and camping motif on the innersole, and a moderate arch contour. Additionally, they include an EVA foam midsole for cushioning and support, and a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern. The approximate weight is 1 lb. 1 oz. per pair."},
 {'query': 'What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, and what materials are used in its construction?',


#### Combine examples

In [19]:
# examples += new_example
examples += d_flattened

In [21]:
# Displaying the first example
print(examples[0])

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?', 'answer': 'Yes'}


In [ ]:
# Testing the QA system with the generated examples
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Yes, the Cozy Cuddles Knit Pullover Set has side pockets.',
 'source_documents': [Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 73}, page_content=": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg. \nImported."),
  Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 503}, page_con

### Manual Evaluation - Fun part

In [ ]:
from langchain.globals import set_debug # to enable debug mode
set_debug(True) # enabling debug mode

In [ ]:
# Testing the QA system with the generated examples
qa.invoke(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set        have side pockets?",
  "context": ": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have 

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Yes, the Cozy Cuddles Knit Pullover Set has side pockets.',
 'source_documents': [Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 73}, page_content=": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg. \nImported."),
  Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 503}, page_con

In [ ]:
# Turning off the debug mode
set_debug(False)

### LLM assisted evaluation

In [25]:
examples += d_flattened

In [26]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What are the key features of the Women's Campside Oxfords, including materials, construction, and fit recommendations?",
  'answer': "The Women's Campside Oxfords are designed with a super-soft canvas material that provides a broken-in feel from the first wear. They feature thick cushioning and quality construction for comfort. The fit recommendation is to order your regular shoe size, and for half sizes not offered, to order up to the next whole size. The shoes have an EVA innersole with Cleansport NXT® antimicrobial odor control, a vintage hunt, fish, and camping motif on the innersole, and a moderate arch contour. Additionally, they include an EVA foam midsole for cushioning and support, and a chain-tread-inspired molded rubber outsole with a modifi

In [28]:
predictions = []  # will store results

# Looping through each example
for ex in examples:
    # Running the chain for each example query
    out = qa.invoke({"query": ex["query"]}) # invoke replaces batch
    # Saving outputs in a consistent structure
    predictions.append({
        "query": ex["query"],
        "result": out.get("result") or out.get("output_text") or out.get("text", ""),
        "source_documents": out.get("source_documents", []),  # for Ragas evaluation
    })

print(f"Generated {len(predictions)} predictions")

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set        have side pockets?",
  "context": ": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have 

In [29]:
predictions

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'result': 'Yes, the Cozy Cuddles Knit Pullover Set has side pockets.',
  'source_documents': [Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 73}, page_content=": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg. \nImported."),
   Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 503}, page

In [30]:
from langchain.evaluation.qa import QAEvalChain

In [ ]:
# Setting up the evaluation chain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # specify the model name

N = min(len(examples), len(predictions)) # specify N based on available data
eval_chain = QAEvalChain.from_llm(llm)

In [ ]:
# Grading the outputs
graded_outputs = eval_chain.evaluate(examples[:N], predictions[:N]) # using N to avoid index errors

[chain/start] [chain:QAEvalChain] Entering Chain run with input:
{
  "input_list": [
    {
      "query": "Do the Cozy Comfort Pullover Set        have side pockets?",
      "answer": "Yes",
      "result": "Yes, the Cozy Cuddles Knit Pullover Set has side pockets."
    },
    {
      "query": "What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?",
      "answer": "The DownTek collection",
      "result": "The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection."
    },
    {
      "query": "What are the key features of the Women's Campside Oxfords, including materials, construction, and fit recommendations?",
      "answer": "The Women's Campside Oxfords are designed with a super-soft canvas material that provides a broken-in feel from the first wear. They feature thick cushioning and quality construction for comfort. The fit recommendation is to order your regular shoe size, and for half sizes not offered, to order up to the next whole

In [33]:
graded_outputs

[{'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'},
 {'results': 'GRADE: CORRECT'}]

In [34]:
# Displaying the first N graded outputs alongside questions, real answers, and predicted answers
for i, eg in enumerate(examples[:N]):
    print(f"Example {i+1}:")
    print("Question:", eg["query"])                       # from examples
    print("Real Answer:", eg["answer"])                   # from examples
    print("Predicted Answer:", predictions[i]["result"])  # from predictions
    # print("Predicted Grade:", graded_outputs[i].get("results") or graded_outputs[i].get("text", ""))
    print()

Example 1:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Cuddles Knit Pullover Set has side pockets.

Example 2:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.

Example 3:
Question: What are the key features of the Women's Campside Oxfords, including materials, construction, and fit recommendations?
Real Answer: The Women's Campside Oxfords are designed with a super-soft canvas material that provides a broken-in feel from the first wear. They feature thick cushioning and quality construction for comfort. The fit recommendation is to order your regular shoe size, and for half sizes not offered, to order up to the next whole size. The shoes have an EVA innersole with Cleansport NXT® antimicrobial odor control, a vintage hunt, fish, and camping 

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [71]:

from langchain_huggingface import HuggingFaceEmbeddings

# Loading the text file
loader = TextLoader("data/nyc_text.txt", encoding="utf-8") # adding encoding to avoid errors

# Creating the embedding model
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",               
    model_kwargs={'device': 'cpu'}                                     
)

# Creating the vector store index
index = VectorstoreIndexCreator(
    embedding=emb,
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80) # specify chunk size and overlap
).from_loaders([loader])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) # specify the model name

# Creating the QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,                                                           
    retriever=index.vectorstore.as_retriever(search_kwargs={"k": 6}), # specify number of documents to retrieve
    return_source_documents=True,
)

/opt/anaconda3/envs/langchain_rag/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/var/folders/xx/q0js_gcd29lfhsshk693bdyw0000gn/T/ipykernel_59891/1131930432.py:13: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a vectorstore when using VectorstoreIndexCreator
  index = VectorstoreIndexCreator(


In [72]:
# Testing it out
question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "How did New York City get its name?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "How did New York City get its name?",
  "context": "New York City has been a metropolitan municipality with a Strong mayor–council form of government since its consolidation in 1898. In New York City, the city government is responsible for public education, correctional institutions, public safety, recreational facilities, sanitation, water supply, and welfare services.\n\nDespite New York's heavy reliance on its vast public transit system, streets are a defining feature of the city. The Commissioners' Plan of 1811 greatly influenced the city's physical development. Several of the city's streets and avenues, including Broadway, Wall Street, Madi

'New York City got its name after it came under British control in 1664. The city was originally named New Amsterdam by Dutch colonists who founded a trading post there. When the British took control, they renamed it New York in honor of King Charles II of England, who granted the lands to his brother, the Duke of York.'

In [73]:
result

{'query': 'How did New York City get its name?',
 'result': 'New York City got its name after it came under British control in 1664. The city was originally named New Amsterdam by Dutch colonists who founded a trading post there. When the British took control, they renamed it New York in honor of King Charles II of England, who granted the lands to his brother, the Duke of York.',
 'source_documents': [Document(id='752e6875-92fe-4189-b98d-bfabb3a8aea1', metadata={'source': 'data/nyc_text.txt'}, page_content='New York City has been a metropolitan municipality with a Strong mayor–council form of government since its consolidation in 1898. In New York City, the city government is responsible for public education, correctional institutions, public safety, recreational facilities, sanitation, water supply, and welfare services.'),
  Document(id='d0c8733d-7a62-48f1-bd44-04faf0d1a620', metadata={'source': 'data/nyc_text.txt'}, page_content="Despite New York's heavy reliance on its vast public

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [74]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "answer": eval_answers[i]}   #"answer" instead of "ground_truths" (ground_truths key is not recognized by QAEvalChain)
    for i, q in enumerate(eval_questions)
]

In [75]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'answer': '8,804,190'},
 {'query': 'Which borough of New York City has the highest population?',
  'answer': 'Brooklyn'},
 {'query': 'What is the economic significance of New York City?',
  'answer': "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."},
 {'query': 'How did New York City get its name?',
  'answer'

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [76]:
# Testing it out
result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Which borough of New York City has the highest population?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Which borough of New York City has the highest population?",
  "context": "New York, often called New York City or NYC, is the most populous city in the United States. With a 2020 population of 8,804,190 distributed over 300.46 square miles (778.2 km2), New York City is the most densely populated major city in the United States and more than twice as populous as Los Angeles, the nation's second-largest city. New York City is located at the southern tip of New York State. It constitutes the geographical and demographic center of both the Northeast megalopolis and the New York metropolitan area, the largest metropolitan area

'Brooklyn has the highest population of all the boroughs in New York City.'

In [77]:
# Updating the keys in the result dictionary
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

# Creating a new dictionary with updated keys
result_updated = {}

for old_key, new_key in key_mapping.items():
    if old_key in result:  # ✅ check before accessing
        if old_key == "source_documents":
            # ✅ Convert Document objects → plain text
            result_updated[new_key] = [doc.page_content for doc in result[old_key]]
        else:
            result_updated[new_key] = result[old_key]

In [17]:
# !pip install --no-cache-dir recordclass

In [158]:
# !pip install ragas==0.1.9

In [78]:
from ragas.integrations.langchain import EvaluatorChain 
# from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,          # CHANGE: use this instead of context_relevancy
    context_recall,
)

# create evaluation chains
faithfulness_chain         = EvaluatorChain(metric=faithfulness)
answer_rel_chain           = EvaluatorChain(metric=answer_relevancy)
context_precision_chain    = EvaluatorChain(metric=context_precision)   # CHANGE: context_precision_chain
context_recall_chain       = EvaluatorChain(metric=context_recall)

1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [82]:
result_updated["retrieved_contexts"] = result_updated.get("contexts", [])
print(result_updated["retrieved_contexts"])

["New York, often called New York City or NYC, is the most populous city in the United States. With a 2020 population of 8,804,190 distributed over 300.46 square miles (778.2 km2), New York City is the most densely populated major city in the United States and more than twice as populous as Los Angeles, the nation's second-largest city. New York City is located at the southern tip of New York State. It constitutes the geographical and demographic center of both the Northeast megalopolis and the New York metropolitan area, the largest metropolitan area in the U.S. by both population and urban area. With over 20.1 million people in its metropolitan statistical area and 23.5 million in its combined statistical area as of 2020, New York is one of the world's most populous megacities, and over 58", "New York City is the most populous city in the United States, with 8,804,190 residents incorporating more immigration into the city than outmigration since the 2010 United States census. More th

**Faithfulness**

In [ ]:
# Rerun your QA chain with a new query
new_question = "Which NYC borough is the most densely populated?"
result = qa_chain.invoke({"query": new_question})

# Build exactly what EvaluatorChain expects
qa_question = result.get("query", "")
qa_answer   = result.get("result", "")
qa_contexts = [d.page_content for d in result.get("source_documents", [])]

payload_for_metric = {
    "question":  qa_question,      
    "answer":    qa_answer,          
    "contexts":  qa_contexts,       
}

# Some internal prompts also look for a single "context" string.
payload_for_metric["context"] = "\n\n".join(qa_contexts)

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Which NYC borough is the most densely populated?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Which NYC borough is the most densely populated?",
  "context": "In 2020, the city had an estimated population density of 29,302.37 inhabitants per square mile (11,313.71/km2), rendering it the nation's most densely populated of all larger municipalities (those with more than 100,000 residents), with several small cities (of fewer than 100,000) in adjacent Hudson County, New Jersey having greater density, as per the 2010 census. Geographically co-extensive with New York County, the borough of Manhattan's 2017 population density of 72,918 inhabitants per square mile (28,154/km2) makes it the highest of any county in the United States

In [ ]:
# Invoking the faithfulness evaluation chain
eval_result = faithfulness_chain.invoke(payload_for_metric)


[chain/start] [chain:EvaluatorChain] Entering Chain run with input:
{
  "question": "Which NYC borough is the most densely populated?",
  "answer": "Manhattan is the most densely populated borough in New York City.",
  "contexts": [
    "In 2020, the city had an estimated population density of 29,302.37 inhabitants per square mile (11,313.71/km2), rendering it the nation's most densely populated of all larger municipalities (those with more than 100,000 residents), with several small cities (of fewer than 100,000) in adjacent Hudson County, New Jersey having greater density, as per the 2010 census. Geographically co-extensive with New York County, the borough of Manhattan's 2017 population density of 72,918 inhabitants per square mile (28,154/km2) makes it the highest of any county in the United States and higher than the density of any individual American city. The next three densest counties in the United States, placing second through fourth, are also New York boroughs: Brooklyn, th

In [94]:
# Displaying the faithfulness score
print("Question:", payload_for_metric["question"])
print("Answer:", payload_for_metric["answer"])
print("Faithfulness:", eval_result["faithfulness"])


Question: Which NYC borough is the most densely populated?
Answer: Manhattan is the most densely populated borough in New York City.
Faithfulness: 0.5


High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [ ]:
fake_result = result.copy()
fake_result["result"] = "we are the champions"

# Building the payload correctly from fake_result
qa_question = fake_result.get("query", "")
qa_answer   = fake_result.get("result", "")
qa_contexts = [d.page_content for d in fake_result.get("source_documents", [])]

# Adding everything the evaluator needs
payload_for_metric = {
    "question": qa_question,       
    "answer": qa_answer,         
    "contexts": qa_contexts,       
    "context": "\n\n".join(qa_contexts),   
    "ground_truth": "Brooklyn has the highest population among NYC boroughs."  # ✅ required for some metrics
}

In [129]:
eval_result = faithfulness_chain.invoke(payload_for_metric)

[chain/start] [chain:EvaluatorChain] Entering Chain run with input:
{
  "question": "Which NYC borough is the most densely populated?",
  "answer": "we are the champions",
  "contexts": [
    "In 2020, the city had an estimated population density of 29,302.37 inhabitants per square mile (11,313.71/km2), rendering it the nation's most densely populated of all larger municipalities (those with more than 100,000 residents), with several small cities (of fewer than 100,000) in adjacent Hudson County, New Jersey having greater density, as per the 2010 census. Geographically co-extensive with New York County, the borough of Manhattan's 2017 population density of 72,918 inhabitants per square mile (28,154/km2) makes it the highest of any county in the United States and higher than the density of any individual American city. The next three densest counties in the United States, placing second through fourth, are also New York boroughs: Brooklyn, the Bronx, and Queens respectively.",
    "New 

In [130]:
print("Question:", payload_for_metric["question"])
print("Answer:", payload_for_metric["answer"])
print("Faithfulness:", eval_result["faithfulness"])

Question: Which NYC borough is the most densely populated?
Answer: we are the champions
Faithfulness: 0.0


**Context Relevancy**

In [107]:
payload_for_metric = {
    "question": qa_question,
    "answer": qa_answer,
    "contexts": qa_contexts,
    "context": "\n\n".join(qa_contexts),
    "ground_truth": "Brooklyn has the highest population among New York City boroughs." # adding ground truth answer for relevancy metric
}

# Run context_recall
eval_result = context_recall_chain.invoke(payload_for_metric)

[chain/start] [chain:EvaluatorChain] Entering Chain run with input:
{
  "question": "Which NYC borough is the most densely populated?",
  "answer": "Manhattan is the most densely populated borough in New York City.",
  "contexts": [
    "In 2020, the city had an estimated population density of 29,302.37 inhabitants per square mile (11,313.71/km2), rendering it the nation's most densely populated of all larger municipalities (those with more than 100,000 residents), with several small cities (of fewer than 100,000) in adjacent Hudson County, New Jersey having greater density, as per the 2010 census. Geographically co-extensive with New York County, the borough of Manhattan's 2017 population density of 72,918 inhabitants per square mile (28,154/km2) makes it the highest of any county in the United States and higher than the density of any individual American city. The next three densest counties in the United States, placing second through fourth, are also New York boroughs: Brooklyn, th

In [108]:
print("Context Recall:", eval_result["context_recall"])

Context Recall: 1.0


High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [ ]:
from ragas import evaluate         
from ragas.metrics import context_recall
from datasets import Dataset      

# Defining question, ground truth, and fake context
question = "Which NYC borough has the highest population?"
ground_truth = "Brooklyn has the highest population among New York City boroughs."
fake_contexts = ["I love Christmas."]  # irrelevant on purpose

# Building a dataset in the expected Ragas format
ds = Dataset.from_dict({
    "question":     [question],
    "answer":       [ground_truth],     # ragas compares this to contexts
    "contexts":     [fake_contexts],    # list[str]
    "ground_truth": [ground_truth],
})

# Evaluating using the Ragas native function
res = evaluate(ds, metrics=[context_recall])


[chain/start] [chain:ragas evaluation] Entering Chain run with input:
{}
[chain/start] [chain:ragas evaluation > chain:row 0] Entering Chain run with input:
{
  "user_input": "Which NYC borough has the highest population?",
  "retrieved_contexts": [
    "I love Christmas."
  ],
  "reference_contexts": null,
  "retrieved_context_ids": null,
  "reference_context_ids": null,
  "response": "Brooklyn has the highest population among New York City boroughs.",
  "multi_responses": null,
  "reference": "Brooklyn has the highest population among New York City boroughs.",
  "rubrics": null
}


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

[chain/start] [chain:ragas evaluation > chain:row 0 > chain:context_recall] Entering Chain run with input:
{
  "user_input": "Which NYC borough has the highest population?",
  "retrieved_contexts": [
    "I love Christmas."
  ],
  "reference": "Brooklyn has the highest population among New York City boroughs."
}
[chain/start] [chain:ragas evaluation > chain:row 0 > chain:context_recall > chain:context_recall_classification_prompt] Entering Chain run with input:
[inputs]
[llm/start] [chain:ragas evaluation > chain:row 0 > chain:context_recall > chain:context_recall_classification_prompt > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: Given a context, and an answer, analyze each sentence in the answer and classify if the sentence can be attributed to the given context or not. Use only 'Yes' (1) or 'No' (0) as a binary classification. Output json with reason.\nPlease return the output in a JSON format that complies with the following schema as specified in JSON

In [156]:
print(res)

{'context_recall': 0.0000}


2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [ ]:
# Building the dataset for Ragas evaluation
from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

# Build dataset for Ragas
data = {
    "question": [ex["query"] for ex in examples],
    "answer": [pred["result"] for pred in predictions],
    "contexts": [[doc.page_content for doc in pred["source_documents"]] for pred in predictions],
    "ground_truth": [
        ex.get("ground_truth", ex.get("answer", ""))
        for ex in examples
    ],
}

dataset = Dataset.from_dict(data)

In [ ]:
# Running faithfulness evaluation
faithfulness_result = evaluate(dataset, metrics=[faithfulness])


[chain/start] [chain:ragas evaluation] Entering Chain run with input:
{}
[chain/start] [chain:ragas evaluation > chain:row 0] Entering Chain run with input:
{
  "user_input": "What is the population of New York City as of 2020?",
  "retrieved_contexts": [
    "New York, often called New York City or NYC, is the most populous city in the United States. With a 2020 population of 8,804,190 distributed over 300.46 square miles (778.2 km2), New York City is the most densely populated major city in the United States and more than twice as populous as Los Angeles, the nation's second-largest city. New York City is located at the southern tip of New York State. It constitutes the geographical and demographic center of both the Northeast megalopolis and the New York metropolitan area, the largest metropolitan area in the U.S. by both population and urban area. With over 20.1 million people in its metropolitan statistical area and 23.5 million in its combined statistical area as of 2020, New Yor

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

[chain/start] [chain:ragas evaluation > chain:row 0 > chain:faithfulness] Entering Chain run with input:
{
  "user_input": "What is the population of New York City as of 2020?",
  "retrieved_contexts": [
    "New York, often called New York City or NYC, is the most populous city in the United States. With a 2020 population of 8,804,190 distributed over 300.46 square miles (778.2 km2), New York City is the most densely populated major city in the United States and more than twice as populous as Los Angeles, the nation's second-largest city. New York City is located at the southern tip of New York State. It constitutes the geographical and demographic center of both the Northeast megalopolis and the New York metropolitan area, the largest metropolitan area in the U.S. by both population and urban area. With over 20.1 million people in its metropolitan statistical area and 23.5 million in its combined statistical area as of 2020, New York is one of the world's most populous megacities, an

In [ ]:
# Printing the faithfulness result
print("Faithfulness:", faithfulness_result)

Faithfulness: {'faithfulness': 0.6286}


In [ ]:
# Running context recall evaluation
context_recall_result = evaluate(dataset, metrics=[context_recall])

[chain/start] [chain:ragas evaluation] Entering Chain run with input:
{}
[chain/start] [chain:ragas evaluation > chain:row 0] Entering Chain run with input:
{
  "user_input": "What is the population of New York City as of 2020?",
  "retrieved_contexts": [
    "New York, often called New York City or NYC, is the most populous city in the United States. With a 2020 population of 8,804,190 distributed over 300.46 square miles (778.2 km2), New York City is the most densely populated major city in the United States and more than twice as populous as Los Angeles, the nation's second-largest city. New York City is located at the southern tip of New York State. It constitutes the geographical and demographic center of both the Northeast megalopolis and the New York metropolitan area, the largest metropolitan area in the U.S. by both population and urban area. With over 20.1 million people in its metropolitan statistical area and 23.5 million in its combined statistical area as of 2020, New Yor

Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

[chain/start] [chain:ragas evaluation > chain:row 0 > chain:context_recall] Entering Chain run with input:
{
  "user_input": "What is the population of New York City as of 2020?",
  "retrieved_contexts": [
    "New York, often called New York City or NYC, is the most populous city in the United States. With a 2020 population of 8,804,190 distributed over 300.46 square miles (778.2 km2), New York City is the most densely populated major city in the United States and more than twice as populous as Los Angeles, the nation's second-largest city. New York City is located at the southern tip of New York State. It constitutes the geographical and demographic center of both the Northeast megalopolis and the New York metropolitan area, the largest metropolitan area in the U.S. by both population and urban area. With over 20.1 million people in its metropolitan statistical area and 23.5 million in its combined statistical area as of 2020, New York is one of the world's most populous megacities, 

In [ ]:
# Printing the context recall result
print("Context Recall:", context_recall_result)

Context Recall: {'context_recall': 0.8333}


### Difficulties Faced During the Lab

This lab was about testing a RAG (Retrieval-Augmented Generation) QA system using **Ragas metrics** like *faithfulness* and *context recall*.  
Here are the main problems I ran into while doing it:

---

#### 1. Import and Version Problems
- Some imports didn’t work, for example `context_relevancy` no longer exists — I had to replace it with `context_precision`.
- Different versions of **LangChain** and **Ragas** caused errors or missing functions.
- The documentation didn’t always match the latest version of the code.

---

#### 2. Wrong Input Keys
- The model output used keys like `query`, `result`, and `source_documents`,  
  but Ragas needed `question`, `answer`, and `contexts`.
- I had to rename and convert the data manually:
  ```python
  [doc.page_content for doc in result["source_documents"]]

### 3. Faithfulness Always 0.0
- At first, the faithfulness score was always 0.0 because the context was not passed correctly.
- The fix was to join all text parts into one string:
     payload["context"] = "\n\n".join(payload["contexts"])
- After that, the score worked (for example, 0.5).

### 4. Context Recall Too High
- Even with random text like "I love Christmas", the score was 1.0 — which made no sense.
- This happened because ground_truth was missing or in the wrong format.
- I fixed it by using the full Ragas evaluate() function with a dataset:
Dataset.from_dict({
    "question": [...],
    "answer": [...],
    "contexts": [...],
    "ground_truth": [...]
})

### 5. Missing Methods
- I tried using .evaluate() on EvaluatorChain, but it doesn’t exist anymore.
- The correct method was .invoke() or using ragas.evaluate() directly.

